# 07 · Level 3 again under the corrected label rule (amendment 1.6), then freeze

Sandesh Lamichhane, Assistant Professor, Yenepoya (Deemed to be University).

**Why this notebook exists.** The external review of notebook 06 made two Level 3 points, and both are correct:

1. Under the 1.5 rules, a host-hour near the end of a period was labelled *negative* even when its 24-hour outcome
   window ran past the period end, so its outcome was not yet known.
2. Because of that, the day-13 hours of the 27 hosts first compromised on day 14 were counted as validation
   negatives and false-alarm host-days. Those same hours are inside the 24-hour windows the test scores for these
   hosts.

**Amendment 1.6** leaves such host-hours out of their period instead of calling them negative (train, val and test
alike). The candidates, grids, selection rule, false-alarm budget and Y = 0.50 do not change. The Level 3 models are
developed again on days 1–13 under the corrected labels and frozen in a new record. The 1.5 freeze
(`freeze_lanl.json`, logistic regression) is kept unchanged as the historical record, and step 2 checks that it is
still intact. The Levels 1–2 freeze is not affected.

No test data is read: development reads LANL days 1–13 only.

**What to expect.** The 1.6 validation numbers are new, so nobody knows them yet. You will see fewer validation
rows than under 1.5, because the day-13 hours of hosts that were not compromised by then are left out. That is the
correction working. Whichever candidate the rule selects is frozen; the 1.5 result is shown beside it.

**Before you start:** *Runtime → Change runtime type → T4 GPU* (needed for the MLP in step 6). High-RAM is not
needed.

**Running it again is safe.** Each candidate resumes where it stopped, and the freeze runs once. After a
disconnect, run step 1 again, then continue from the step that was interrupted.

| Step | What it does | Approximate time |
|---|---|---|
| 1 | connect Drive and GitHub, update the code, run the tests (expected `… passed`, no failures) | 4 min |
| 2 | check that both existing freezes are intact (expected three `OK` lines) | 5–20 min |
| 3 | develop logistic regression under 1.6 | 15–30 min |
| 4 | develop random forest under 1.6 | 40–70 min |
| 5 | develop XGBoost under 1.6 | 10–20 min |
| 6 | develop the MLP under 1.6 (GPU) | 5–10 min |
| 7 | development table, with the 1.5 results beside it | 1 min |
| 8 | freeze Level 3 under 1.6, then check the freeze (expected `OK Level 3 (rules 1.6): …`) | 2–10 min |

**Where the results go.** Everything is saved in the Drive copy of the repository
(`MyDrive/PhD-AI-Cyber/code/phd-ai-threat-detection/`) and on GitHub. There is nothing to move by hand.

* `results/models/development/lanl_1.6/<candidate>.json`: the development records under 1.6
* `results/models/development/lanl_1.6_summary.md`: the development table, with the 1.5 results beside it
* `results/models/freeze_lanl_1.6.json`: the new Level 3 freeze record (the one H2 is scored with)
* `results/models/freeze_summary.md`: all freezes in words (the 1.5 one marked as superseded)
* `results/models/logs/<time>_lanl_….log`: run logs

The new model files go to `MyDrive/PhD-AI-Cyber/models/development/lanl_1.6/` (Drive only). The 1.5 model files in
`models/development/lanl/` stay where they are. **Do not open, move or re-save any model file.**

**When it finishes:** use *File → Download → Download .ipynb* and send the finished notebook to the project
thread. Then run notebook 08.

## Step 1 · Connect and update (expected end: `in sync with GitHub at …` and `… passed` with no failures)

Always run this cell first, also after a disconnect. It loads the newest code.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import datetime as _dt, json, os, pathlib, shutil, subprocess, time

DRIVE_ROOT = '/content/drive/MyDrive/PhD-AI-Cyber'
REPO_DIR   = f'{DRIVE_ROOT}/code/phd-ai-threat-detection'
os.environ['PHD_DRIVE_ROOT'] = DRIVE_ROOT
os.environ['PHD_SCRATCH'] = '/content/scratch'
TOKEN  = userdata.get('GITHUB_TOKEN')
REMOTE = f'https://{TOKEN}@github.com/sandesh20lamichhane/phd-ai-threat-detection.git'

# Git keeps its database on the Colab disk, never on Drive: Drive's file system can crash git while it unpacks
# downloads ("unpack-objects died of signal 7"). The files themselves stay in the Drive folder, so every result
# is in Drive the moment it is written, and the sync below copies it to GitHub.
GIT_DIR = '/content/phd-repo.git'
STATE = pathlib.Path(DRIVE_ROOT, 'code', '.phd_sync_state.json')    # commit the Drive folder last matched
os.environ['GIT_DIR'], os.environ['GIT_WORK_TREE'] = GIT_DIR, REPO_DIR
RECORD_DIRS = ('results/', 'docs/data_cards/', 'config/feature_mapping/')
_trust_drive_records = True

def _mask(text):
    return text.replace(TOKEN or '\0', '***').strip()

def git(*args, check=False):
    r = subprocess.run(['git', *args], capture_output=True, text=True, cwd=REPO_DIR)
    out = _mask(r.stdout + r.stderr)
    if check and r.returncode != 0:
        print(out)
        raise SystemExit(f'git {args[0]} failed (see message above)')
    return r.returncode, out

def _stamp():
    return _dt.datetime.now(_dt.timezone.utc).strftime('%Y%m%dT%H%M%SZ')

def _last_synced_commit():
    """The commit the Drive folder was last synced to: from the state file, else from the old .git on Drive."""
    try:
        if STATE.exists():
            return json.loads(STATE.read_text()).get('commit')
        g = pathlib.Path(REPO_DIR, '.git')
        head = (g / 'HEAD').read_text().strip()
        if not head.startswith('ref: '):
            return head
        ref = head[5:]
        if (g / ref).exists():
            return (g / ref).read_text().strip()
        for line in (g / 'packed-refs').read_text().splitlines():
            if line.endswith(' ' + ref):
                return line.split()[0]
    except (OSError, ValueError):
        pass
    return None

def setup_git():
    """Once per Colab session: download the git database to the Colab disk and line it up with the Drive folder."""
    global _trust_drive_records
    if pathlib.Path(GIT_DIR, 'HEAD').exists():
        return
    pathlib.Path(REPO_DIR).mkdir(parents=True, exist_ok=True)
    env = {k: v for k, v in os.environ.items() if k not in ('GIT_DIR', 'GIT_WORK_TREE')}
    r = subprocess.run(['git', 'clone', '--bare', '--quiet', REMOTE, GIT_DIR], capture_output=True, text=True, env=env)
    if r.returncode != 0:
        print(_mask(r.stdout + r.stderr))
        raise SystemExit('git clone failed (see message above)')
    git('config', 'core.bare', 'false', check=True)
    git('config', 'core.filemode', 'false')
    old = _last_synced_commit()
    known = bool(old) and git('cat-file', '-e', f'{old}^{{commit}}')[0] == 0
    git('reset', '-q', old if known else 'HEAD', check=True)      # sets the index only; Drive files untouched
    _trust_drive_records = known
    if not known:
        print('note: could not tell which commit the Drive folder last matched; GitHub wins for every tracked '
              'file, and Drive versions that differ are backed up first')
    dot = pathlib.Path(REPO_DIR, '.git')
    if dot.exists():
        dest = pathlib.Path(DRIVE_ROOT, 'backups', f'old_drive_git_{_stamp()}')
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(dot), str(dest))
        print(f'old git folder on Drive moved to {dest} (no longer used)')
    print('git database on the Colab disk; Drive folder last matched ' + (old[:7] if known else 'an unknown commit'))

def _push_hint(out):
    o = out.lower()
    if any(k in o for k in ('authentication failed', 'invalid username', 'invalid credentials', '403', 'permission')):
        return ('GitHub did not accept the token. If it has expired, create a new fine-grained token (repository '
                'phd-ai-threat-detection, Contents: Read and write) and save it as the Colab secret GITHUB_TOKEN.')
    if 'rejected' in o or 'fetch first' in o or 'non-fast-forward' in o:
        return 'GitHub had a newer commit; fetching it and trying again.'
    return 'GitHub or the network did not respond; trying again in a moment.'

def sync(message=None, strict=True, also=()):
    """Make the Drive copy of the repo identical to GitHub, keeping every result produced here.

    Never merges or rebases, so open notebooks or other local edits cannot block it:
    1. remember result files changed here (results/, docs/data_cards/, config/feature_mapping/)
    2. back up every other local change to MyDrive/PhD-AI-Cyber/backups/sync_<time>/
    3. reset the copy to GitHub's main, put the result files back, commit and push them
    Results are on Drive before this runs. If GitHub cannot be reached, strict=False only warns: the next
    sync (the next step, or step 1 in a new session) saves the same files.
    `also` names other files changed here on purpose (e.g. docs/protocol/prior_exposure.md), saved like results.
    """
    global _trust_drive_records
    setup_git()
    git('config', 'user.name', 'Sandesh Lamichhane')
    git('config', 'user.email', 'san30lamichhane@gmail.com')
    tries, out = 5, ''
    for attempt in range(1, tries + 1):
        code, out = git('fetch', '--quiet', REMOTE, 'main')
        if code == 0:
            base = git('merge-base', 'HEAD', 'FETCH_HEAD')[1] or 'FETCH_HEAD'
            changed = set(git('diff', '--name-only', base)[1].splitlines())
            untracked = set(git('ls-files', '--others', '--exclude-standard')[1].splitlines())
            # untracked non-record files survive the reset unless GitHub has a file at the same path
            untracked = {f for f in untracked if f.startswith(RECORD_DIRS) or git('cat-file', '-e', f'FETCH_HEAD:{f}')[0] == 0}
            changed |= untracked
            is_file = lambda f: pathlib.Path(REPO_DIR, f).is_file()
            keep = sorted(f for f in changed if is_file(f) and (f in also or f.startswith(RECORD_DIRS)
                          and (_trust_drive_records or f in untracked)))
            other = sorted(f for f in changed if f not in keep and is_file(f))
            stamp = _stamp()
            hold = pathlib.Path('/content/sync_hold') / stamp
            for f in keep:
                (hold / f).parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(pathlib.Path(REPO_DIR, f), hold / f)
            if other:
                bak = pathlib.Path(DRIVE_ROOT, 'backups', f'sync_{stamp}')
                for f in other:
                    (bak / f).parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(pathlib.Path(REPO_DIR, f), bak / f)
                print(f'{len(other)} other local change(s) backed up to {bak} (e.g. notebook outputs)')
            git('reset', '--hard', '--quiet', 'FETCH_HEAD', check=True)
            for f in keep:
                dest = pathlib.Path(REPO_DIR, f)
                dest.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(hold / f, dest)
            if keep:
                git('add', '--', *keep, check=True)
                if git('diff', '--cached', '--quiet')[0] != 0:
                    git('commit', '-q', '-m', message or 'Save results produced in Colab', check=True)
            code, out = git('push', '--quiet', REMOTE, 'HEAD:main')
            if code == 0:
                head = git('log', '--oneline', '-1')[1]
                STATE.write_text(json.dumps({'commit': git('rev-parse', 'HEAD')[1], 'utc': stamp}))
                _trust_drive_records = True
                print(f'in sync with GitHub at {head}' + (f'; {len(keep)} result file(s) saved' if keep else ''))
                return True
        print(f'GitHub refused the save (try {attempt} of {tries}): {out or "no message"}')
        print('  ' + _push_hint(out))
        if attempt < tries:
            time.sleep(min(60, 4 * 2 ** (attempt - 1)))
    msg = ('Could not save to GitHub. Nothing is lost: every result is in the Drive folder, and the next save '
           '(the next step, or step 1 after reopening the notebook) sends it to GitHub. Copy the messages above '
           'into the project thread if this keeps happening.')
    if strict:
        raise SystemExit(msg)
    print('WARNING: ' + msg)
    return False


sync('Save results from an earlier Colab run')      # update the code; results made here are kept
assert pathlib.Path(REPO_DIR, 'config/models_lanl_1.6.yaml').exists(), 'old code: run this cell again'
%cd $REPO_DIR
!pip -q install -r requirements.txt
!python -m pytest -q tests
import sys
sys.path.insert(0, REPO_DIR)
from IPython.display import Markdown, display

def push(message, also=()):
    """Save results from the Drive copy of the repo to GitHub, so both hold the same records.
    If GitHub cannot be reached it only warns: the results are in Drive and the next save sends them."""
    if sync(message, strict=False, also=also):
        print('saved to GitHub:', message)

def develop(*args, message):
    """Run scripts/develop.py (its output shows below); push the records only if it succeeded."""
    ip = get_ipython()
    ip.system('python scripts/develop.py ' + ' '.join(args))
    ok = ip.user_ns.get('_exit_code', 1) == 0
    push(message if ok else message + ' (FAILED, log only)')
    if not ok:
        raise SystemExit('This step failed. Copy the error above into the project thread.')

## Step 2 · Check the existing freezes (expected: three `OK` lines)

Expected: `OK Levels 1-2: random_forest …`, `OK Level 3 (rules 1.5, superseded): logreg …` and
`OK prior-exposure declaration with the amendment 1.6 addendum`. Nothing is changed here.

In [ ]:
from src.models import seal, lanl_develop as L
for track, label in (('l12', 'Levels 1-2'), ('lanl_1.5', 'Level 3 (rules 1.5, superseded)')):
    f = seal.require_frozen(track=track)
    print(f"OK {label}: {f['proposed']}, frozen {f['frozen_utc']}; {len(f['model_files_sha256'])} model files and "
          f"{len(f['development_records_sha256'])} development records match their SHA-256")
pe = pathlib.Path(REPO_DIR, 'docs/protocol/prior_exposure.md').read_text()
assert 'Author to complete' not in pe and 'Addendum (amendment 1.6' in pe, 'old prior_exposure.md: run step 1 again'
print('OK prior-exposure declaration with the amendment 1.6 addendum')
cfg = L.load_config()
print(f"Level 3 rules {cfg['version']}: unresolved host-hours at a period end -> {cfg['label']['unresolved_horizon']}")
assert str(cfg['version']) == '1.6' and cfg['label']['unresolved_horizon'] == 'exclude'
if pathlib.Path(REPO_DIR, 'results/models/freeze_lanl_1.6.json').exists():
    print('Level 3 is already frozen under 1.6: steps 3-7 only show the records, step 8 checks the freeze.')

## Step 3 · Develop logistic regression under the 1.6 rules (15–30 min)

Expected end: `[lanl/logreg] done: val early-alert rate …`, then the table and `in sync with GitHub at …`.

In [ ]:
develop('--lanl', '--candidate', 'logreg', message='Develop logreg for Level 3 under amendment 1.6 (LANL days 1-13)')

## Step 4 · Develop random forest under the 1.6 rules (40–70 min)

Expected end: `[lanl/random_forest] done: val early-alert rate …`, then the table and `in sync with GitHub at …`.

In [ ]:
develop('--lanl', '--candidate', 'random_forest', message='Develop random_forest for Level 3 under amendment 1.6 (LANL days 1-13)')

## Step 5 · Develop XGBoost under the 1.6 rules (10–20 min)

Expected end: `[lanl/xgboost] done: val early-alert rate …`, then the table and `in sync with GitHub at …`.

In [ ]:
develop('--lanl', '--candidate', 'xgboost', message='Develop xgboost for Level 3 under amendment 1.6 (LANL days 1-13)')

## Step 6 · Develop the MLP under the 1.6 rules (GPU, 5–10 min)

Expected end: `[lanl/mlp] done: val early-alert rate …`, then the table and `in sync with GitHub at …`.

In [ ]:
develop('--lanl', '--candidate', 'mlp', message='Develop mlp for Level 3 under amendment 1.6 (LANL days 1-13)')

## Step 7 · Development table, with the 1.5 results beside it (expected: four candidates `complete yes`, 5 seeds each)

In [ ]:
develop('--lanl', '--status', message='Level 3 development table under 1.6, with the 1.5 results beside it')
bad = [r['candidate'] for r in L.summary_table() if not r['complete'] or r['seeds_done'] != 5]
if bad:
    raise SystemExit(f'development is not complete for {bad}: run its step again')
print('all 4 Level 3 candidates complete with 5 seeds under rules 1.6')
display(Markdown(pathlib.Path(REPO_DIR, 'results/models/development/lanl_1.6_summary.md').read_text()))

## Step 8 · Freeze Level 3 under 1.6 and check it (expected: `OK Level 3 (rules 1.6): …`)

The preregistered rule picks the candidate with the best seed-mean val early-alert rate (ties: val PR-AUC, then
config order). It runs once, writes `results/models/freeze_lanl_1.6.json` with the SHA-256 of every model file,
development record and the prior-exposure file, names the 1.5 freeze it supersedes, and is saved to GitHub before
the check.

In [ ]:
FREEZE_L3 = pathlib.Path(REPO_DIR, 'results/models/freeze_lanl_1.6.json')
if FREEZE_L3.exists():
    print('Level 3 is already frozen under 1.6 (the freeze runs once); showing the record.')
else:
    develop('--lanl', '--freeze', message='Freeze Level 3 under amendment 1.6: predictive model chosen on LANL validation by the preregistered rule')
sync('Save the Level 3 freeze (rules 1.6)', strict=True)      # stops here if GitHub does not have the freeze yet
import importlib
importlib.reload(seal)
f = seal.require_frozen(track='lanl')
print(f"OK Level 3 (rules {f['rules_version']}): {f['proposed']}, frozen {f['frozen_utc']}, committed to git "
      f"unchanged; {len(f['model_files_sha256'])} model files and {len(f['development_records_sha256'])} development "
      f"records match their SHA-256; supersedes {f['supersedes']['file']} ({f['supersedes']['sha256'][:12]}…)")
display(Markdown(pathlib.Path(REPO_DIR, 'results/models/freeze_summary.md').read_text()))
print('Done. Download this notebook (File -> Download -> Download .ipynb) and send it to the project thread.')